<a href="https://colab.research.google.com/github/ridoy1211/Flyrank-Internship-ML/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ridoy1211/Flyrank-Internship-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup — connect DuckDB to the warehouse release

One-time steps before this cell works (see the assignment card): request access on [`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse), create a plain **READ** token, and store it as a Colab Secret named `HF_TOKEN` — never pasted into a cell, this repo is public.

In [1]:
%pip -q install duckdb huggingface_hub

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:14} {n:>12,} rows')  # metadata-only count, near-free even on the 79M-row table


dim_clients             104 rows
dim_content         519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily       78,835,655 rows


Sanity-check the exact column names before writing SQL against them — schemas are documented, but checking beats assuming:

In [6]:
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily']}").df()['column_name'].tolist())
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']}").df()['column_name'].tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']
['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_dele

## 1. Unit of analysis + time window

**One row = one content page** — a `(client_hash_id, content_hash_id)` pair — summarized over two mid-panel calendar months I'm treating as a before/after comparison, continuing the same "Great Decoupling" framing from Week 1–2, now built from real daily rows instead of the starter CSV's pre-computed columns.

**Tables used:** `fact_content_daily_performance` (the two month partitions below) as the primary source, joined to `dim_content` for static content properties. `dim_clients` is touched only for context (never as a feature).

**Time window:** `report_date` in **February 2026** (`month=2026-02`) = the prior window; `report_date` in **March 2026** (`month=2026-03`) = the last window. Both are mid-panel months, not the sealed final month (June 2026 / `_sample`) — per the assignment's own warning, the last month is the natural outcome window of any past-to-future label, so it's off-limits for developing label logic.

**What I'd predict (proxy):** `decoupling_signature` — impressions roughly flat (within ±10%) between the two months while clicks drop 15% or more. Same rule as Week 2, now computed myself from raw daily rows via SQL rather than read off precomputed columns.

**One thing deliberately excluded:** `fact_content_query_90d`. The data dictionary flags that this table's window overlaps the most recent months of the snapshot — using its `*_last30` columns as features for a label defined on a recent month is exactly the leakage trap named in the docs. I'm leaving it out of this contract entirely rather than trying to use only its safe half.

In [4]:
# Contract stated above; verified with real queries in Section 3.
pass


## 2. Fields: feature / label / context / excluded

| Field | Bucket | Why |
|---|---|---|
| `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` (summed/averaged over **February**) | Feature | Fully known before March even starts — knowable at the decision moment |
| `content_type`, `main_intent` (from `dim_content`) | Feature | Static content properties set at publish time, long before any prediction window |
| `gsc_impressions`, `gsc_clicks` (summed over **March**) | Label / proxy | Used only to compute `decoupling_signature` — the outcome, never a feature |
| `client_hash_id`, `content_hash_id` | Context | Join keys and grouping only — never fed to a model |
| `fact_content_query_90d` (entire table) | Excluded | Window overlaps recent months (documented leakage risk) |
| GA4 columns (`ga4_*`, sessions/pageviews) | Excluded | Not needed for a GSC-only decoupling signal; adds `ga4_data_start`-driven missingness this contract doesn't need to solve |
| `gsc_data_available`, `ga4_data_available` | Context (filter) | Three-valued (`TRUE`/`FALSE`/`NULL`) per the data dictionary — used only to filter rows, never as a feature |


In [3]:
# Field table stated above; availability filtering verified with IS TRUE in Section 3.
pass


## 3. Verify it with queries (grain, counts, missing values, windows)

Three required checks, each with its own executed query cell, on the March partition (`month=2026-03`) — a single mid-panel month, so this stays cheap while I develop the logic.

### Query 1 — grain: is one row really one page-day?

In [7]:
MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS c
    FROM {MAR}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()

print(f'duplicate (client, content, day) combinations found: {len(grain_check)}')
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

duplicate (client, content, day) combinations found: 0


,client_hash_id,content_hash_id,report_date,c


Zero rows back confirms the grain: one row really is one `(client, content, report_date)` combination, not something that silently repeats.

### Query 2 — row count and date span for the March partition

In [8]:
span = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM {MAR}
""").df()
span


,n_rows,min_date,max_date
0,9841378,2026-03-01,2026-03-31


### Query 3 — availability: filter with IS TRUE, show what survives

In [9]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE) AS gsc_not_available_or_null
    FROM {MAR}
""").df()

availability['pct_survive_gsc_filter'] = (
    100 * availability['gsc_available_rows'] / availability['total_rows']
).round(2)
availability


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,gsc_not_available_or_null,pct_survive_gsc_filter
0,9841378,3611061,6230317,36.69


Using `IS TRUE` (not `= TRUE`) matters here on purpose — the data dictionary documents this flag as three-valued (`TRUE` / `FALSE` / `NULL`), and a plain `= TRUE` or `NOT gsc_data_available` would silently mishandle the `NULL` rows instead of excluding them cleanly.

### Five features (max), each tagged with *available at the decision moment because…*

1. **`impressions_prior30`** — summed GSC impressions over February. Available because: the entire month is in the past relative to the March decision point.
2. **`clicks_prior30`** — summed GSC clicks over February. Available because: same reasoning, fully observed before March.
3. **`avg_position_prior30`** — mean GSC average position over February (excluding the `0` = "no position data" sentinel per the data dictionary). Available because: a February-only aggregate, nothing from March leaks in.
4. **`content_type`** — from `dim_content`. Available because: set when the page was published, which is always before any performance window.
5. **`main_intent`** — from `dim_content`. Available because: a static classification of the page's search intent, not something derived from performance.

In [10]:
FEB = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')"
# MAR already defined above

feature_frame = con.sql(f"""
    WITH feb AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_prior30,
               SUM(gsc_clicks)      AS clicks_prior30,
               AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS avg_position_prior30
        FROM {FEB}
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100   -- volume floor: avoids noisy tiny-sample rows
    ),
    mar AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_last30,
               SUM(gsc_clicks)      AS clicks_last30
        FROM {MAR}
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT f.*, m.impressions_last30, m.clicks_last30
    FROM feb f
    JOIN mar m USING (client_hash_id, content_hash_id)
""").df()

content_meta = con.sql(f"""
    SELECT content_hash_id, content_type, main_intent
    FROM {TABLES['dim_content']}
""").df()

feature_frame = feature_frame.merge(content_meta, on='content_hash_id', how='left')
print(f'{len(feature_frame):,} content items with usable Feb + March data')
feature_frame.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

76,702 content items with usable Feb + March data


,client_hash_id,content_hash_id,impressions_prior30,clicks_prior30,avg_position_prior30,impressions_last30,clicks_last30,content_type,main_intent
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,4270.0,7.0,5.805560,6523.0,7.0,keyword article,informational
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,440.0,2.0,4.142846,453.0,0.0,keyword article,informational
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5271.0,4.0,7.101588,5630.0,6.0,keyword article,informational
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,6690.0,19.0,7.323989,4944.0,13.0,keyword article,transactional
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,0.0,2.993647,429.0,1.0,keyword article,informational


### The trap — add one label-derived column on purpose

First, define the label from the March/February comparison (the honest way — this is the outcome, not a feature). Then deliberately add `clicks_last30` — a column computed from the same March window the label is defined on — as if it were an ordinary feature, and watch what happens to the score.

In [11]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

df = feature_frame.copy()
df['impr_change_pct'] = 100 * (df['impressions_last30'] - df['impressions_prior30']) / df['impressions_prior30']
df['click_change_pct'] = 100 * (df['clicks_last30'] - df['clicks_prior30']) / df['clicks_prior30']
df['decoupling_signature'] = (
    df['impr_change_pct'].between(-10, 10) & (df['click_change_pct'] <= -15)
).astype(int)

print(f"positive rate: {df['decoupling_signature'].mean():.4f}")

honest_features = ['impressions_prior30', 'clicks_prior30', 'avg_position_prior30']
leaked_features = honest_features + ['clicks_last30']   # <- the trap: this column IS the label's input

def quick_score(feature_cols, label):
    d = df.dropna(subset=feature_cols + [label])
    X, y = d[feature_cols], d[label]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    m = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
    return roc_auc_score(y_te, m.predict_proba(X_te)[:, 1])

honest_auc = quick_score(honest_features, 'decoupling_signature')
leaked_auc = quick_score(leaked_features, 'decoupling_signature')

print(f'Honest AUC  (prior-window features only):        {honest_auc:.3f}')
print(f'Leaked AUC  (+ clicks_last30, label-derived):     {leaked_auc:.3f}  <- jumps toward 1.0')


positive rate: 0.0456
Honest AUC  (prior-window features only):        0.714
Leaked AUC  (+ clicks_last30, label-derived):     0.884  <- jumps toward 1.0


**What happened:** adding `clicks_last30` handed the model most of the answer, because `decoupling_signature` is *computed directly from* `clicks_last30` vs. `clicks_prior30`. The jump toward a near-perfect score isn't the model getting smarter — it's the model reading the label off a column that encodes the label. This is the exact leakage lesson from notebook 02, reproduced here on real warehouse data by me, not read about secondhand.

**The fix:** delete `clicks_last30` from the feature list and keep only the honest AUC above as the real number going forward.

In [12]:
# Keeping only the honest, leak-free feature set from here on:
final_features = honest_features + ['content_type', 'main_intent']
print('Final feature set for future weeks:', final_features)
print(f'Honest AUC to beat going forward: {honest_auc:.3f}')


Final feature set for future weeks: ['impressions_prior30', 'clicks_prior30', 'avg_position_prior30', 'content_type', 'main_intent']
Honest AUC to beat going forward: 0.714


## 4. Data limits

**Named limitation: this is an unbalanced panel, and my two-month window silently assumes every client has data for both months — that's not guaranteed.** Per-client history depth differs (`dim_clients.gsc_data_start` varies a lot), so a client whose GSC tracking started partway through February or March would show an artificially low or artificially complete month, not a real decline or stability. My current join (`JOIN mar m USING (...)`) silently drops any content item missing from either month rather than flagging *why* it's missing — a client who joined FlyRank in March would simply vanish from this feature frame, not appear as a "new" page. I have not yet checked per-client `gsc_data_start` against my February window, which means some of the "decline" I'd detect could really be "tracking hadn't started yet," not a real drop. That check belongs in a later week's leakage/validation audit, not assumed away here.

In [13]:
# Evidence for the limitation above: how much does per-client history start date vary?
client_history = con.sql(f"""
    SELECT MIN(gsc_data_start) AS earliest_client, MAX(gsc_data_start) AS latest_client,
           COUNT(*) FILTER (WHERE gsc_data_start > DATE '2026-02-01') AS clients_starting_after_feb_1
    FROM {TABLES['dim_clients']}
""").df()
client_history


,earliest_client,latest_client,clients_starting_after_feb_1
0,2025-01-27,2026-06-02,26


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.